# 🏥 Benchmark Evaluations Notebook 📈
  
**Goal**: Load ranking results for different retrieval methods, generate query/response pairs, compute per-query metrics (NDCG@10, NDCG@20, Recall@10), aggregate averages, select the best method, and upload to Azure AI Foundry.  
Phases:  
1. 📦 **Preprocess** (load data & build pairs)  
2. ⚙️ **Evaluate** (compute metrics)  
3. ➡️ **Post-process** (aggregate & upload)  

In [8]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import pandas as pd
import json
from azure.identity import DefaultAzureCredential
from azure.ai.evaluation import evaluate, ContentSafetyEvaluator
from azure.ai.projects import AIProjectClient
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    Evaluation, Dataset, EvaluatorConfiguration, ConnectionType
)
from azure.ai.evaluation._evaluate._eval_run import EvalRun
import src.evals.sdk.custom_azure_ai_evaluations as custom_eval
from azure.ai.evaluation import F1ScoreEvaluator, RelevanceEvaluator, ViolenceEvaluator
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import ServiceResponseError
import time
import subprocess

load_dotenv()

EvalRun._start_run = custom_eval.custom_start_run

def _generate_custom_tags(case_id: str, git_commit: str, class_name: str):
    tags = [
        ("case", case_id),
        ("commit", git_commit),
        ("class", class_name),
    ]
    custom_eval.CUSTOM_TAGS = tags
    return tags

AZURE_AI_FOUNDRY_CONNECTION_STRING = os.getenv("AZURE_AI_FOUNDRY_CONNECTION_STRING")
_, AZ_SUBSCRIPTION_ID, AZ_RESOURCE_GROUP, AI_FOUNDRY_PROJECT_NAME  = AZURE_AI_FOUNDRY_CONNECTION_STRING.split(';')

azure_ai_project = {
    "subscription_id": AZ_SUBSCRIPTION_ID,
    "resource_group_name": AZ_RESOURCE_GROUP,
    "project_name": AI_FOUNDRY_PROJECT_NAME,
}

credential = DefaultAzureCredential()

ai_project_client = AIProjectClient.from_connection_string(
    credential=DefaultAzureCredential(),
    conn_str=AZURE_AI_FOUNDRY_CONNECTION_STRING,
)

print(f"Connected to Azure AI Project: {azure_ai_project['project_name']}")

git_hash = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"🔖 Using commit: {git_hash}")


Connected to Azure AI Project: medevals
🔖 Using commit: a9c4f3a


In [ ]:
EVAL_DIR      = Path("../evals/benchmark/medindexer")
queries_path  = EVAL_DIR / "queries.jsonl"
qrels_path    = EVAL_DIR / "qrels.jsonl"
ranking_files = {
    "hybrid_semantic": EVAL_DIR / "rankings-hybrid-semantic.jsonl",
    "hybrid":          EVAL_DIR / "rankings-hybrid.jsonl",
    "keyword":         EVAL_DIR / "rankings-keyword.jsonl",
    "vector":          EVAL_DIR / "rankings-vector.jsonl",
}

# load queries into dict
queries = {}
with open(queries_path, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        queries[r["id"]] = r["query"]

# load qrels into dict of dicts
qrels = {}
with open(qrels_path, "r", encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        qrels.setdefault(r["query"], {})[r["document"]] = int(r["relevant"])

# for each ranking method, build eval_data_<method>.jsonl
for method, fp in ranking_files.items():
    out_path = EVAL_DIR / f"eval_data_{method}.jsonl"
    with open(fp, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            rec = json.loads(line)
            qid = rec["query"]
            sorted_docs = [doc for doc, _ in sorted(
                rec["ranking"].items(), key=lambda x: x[1], reverse=True
            )]
            record = {
                "query_id":     qid,
                "query":        queries.get(qid, ""),
                "method":       method,
                "ranking":      json.dumps(rec["ranking"]),
                "ground_truth": json.dumps(qrels.get(qid, {}))
            }
            fout.write(json.dumps(record) + "\n")
    print(f"📦 Preprocessed data for '{method}' → {out_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'evals/benchmark/medindexer/queries.jsonl'

In [3]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [4]:
# ─── 2. Custom Evaluators ───────────────────────────────────────────────────────
from src.evals.custom.ndcg_evaluator import NDCGEvaluator
from src.evals.custom.search_recall_evaluator import SearchRecallEvaluator

In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import evaluate

# setup
load_dotenv()
AZ_CONN = os.getenv("AZURE_AI_FOUNDRY_CONNECTION_STRING")
credential = DefaultAzureCredential()
ai_project_client = AIProjectClient.from_connection_string(credential=credential, conn_str=AZ_CONN)

# quick single-record test
single = Path("../evals/benchmark/medindexer/single.jsonl")
with open(single, "w") as f:
    f.write(open(Path("../evals/benchmark/medindexer/eval_data.jsonl")).readline())


In [6]:
res = evaluate(
    data=str(single),
    evaluators={
        "ndcg_3":    NDCGEvaluator(3),
        "ndcg_10":   NDCGEvaluator(10),
        "recall_10": SearchRecallEvaluator(10),
    },
    azure_ai_project=None,
    evaluator_config={
        "ndcg_3":    {"column_mapping": {"ranking":"${data.ranking}","ground_truth":"${data.ground_truth}","query_id":"${data.query_id}"}},
        "ndcg_10":   {"column_mapping": {"ranking":"${data.ranking}","ground_truth":"${data.ground_truth}","query_id":"${data.query_id}"}},
        "recall_10": {"column_mapping": {"ranking":"${data.ranking}","ground_truth":"${data.ground_truth}","query_id":"${data.query_id}"}},
    }
)
print("Single rec metrics:", res)

[2025-04-25 14:54:54 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 14:54:54 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 14:54:54 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 14:54:54 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_ndcg_3_20250425_145453_976262, log path: /Users/marcjimz/.promptflow/.runs/azure_ai_evaluation_evaluators_ndcg_3_20250425_145453_976262/logs.txt
[2025-04-25 14:54:54 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_ndcg_10

2025-04-25 14:54:54 -0700   13228 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-04-25 14:54:54 -0700   13228 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-04-25 14:54:54 -0700   13228 execution.bulk     INFO     The timeout for the batch run is 3600 seconds.
2025-04-25 14:54:54 -0700   13228 execution.bulk     INFO     Current system's available memory is 8851.546875MB, memory consumption of current process is 307.875MB, estimated available worker count is 8851.546875/307.875 = 28
2025-04-25 14:54:54 -0700   13228 execution.bulk     INFO     Set process count to 1 by taking the minimum value among the factors of {'default_worker_count': 4, 'row_count': 1, 'estimated_worker_count_based_on_memory_usage': 28}.
2025-04-25 14:54:55 -0700   13228 execution.bulk     INFO     Process name(SpawnProcess-4)-Process id(13358)-Line number(0) start execution.


In [7]:
res

{'rows': [{'inputs.query_id': 'q1',
   'inputs.query': "Crohn's disease Adalimumab prior authorization criteria",
   'inputs.method': 'hybrid-semantic',
   'inputs.ranking': '{"d40": 2.8734118938446045, "d52": 2.8266220092773438, "d32": 2.811025381088257, "d66": 2.753601312637329, "d17": 2.7202813625335693, "d14": 2.7110650539398193, "d51": 2.7039756774902344, "d18": 2.6671109199523926, "d2": 2.6607306003570557, "d71": 2.613231658935547, "d50": 2.594090223312378, "d34": 2.579911470413208, "d1": 2.557225465774536, "d41": 2.419337034225464, "d25": 2.4127793312072754, "d3": 2.397005558013916, "d7": 2.3759145736694336, "d15": 2.363330841064453, "d31": 2.335416316986084, "d44": 2.3316946029663086, "d21": 2.3296563625335693, "d67": 2.2875630855560303, "d8": 2.1701009273529053, "d60": 2.0933139324188232, "d23": 2.0787806510925293, "d33": 1.913864016532898, "d56": 1.89064621925354, "d48": 1.8397799730300903, "d61": 1.8128403425216675, "d57": 1.6710525751113892, "d27": 1.6526201963424683, "d28"

In [9]:
out = evaluate(
    evaluation_name="search-benchmark",
    data=str(eval_data),
    evaluators={
        "ndcg_3":    NDCGEvaluator(3),
        "ndcg_10":   NDCGEvaluator(10),
        "recall_10": SearchRecallEvaluator(10),
    },
    azure_ai_project=None,
    evaluator_config={
        "ndcg_3":    {"column_mapping": {"ranking":"${data.ranking}","ground_truth":"${data.ground_truth}","query_id":"${data.query_id}"}},
        "ndcg_10":   {"column_mapping": {"ranking":"${data.ranking}","ground_truth":"${data.ground_truth}","query_id":"${data.query_id}"}},
        "recall_10": {"column_mapping": {"ranking":"${data.ranking}","ground_truth":"${data.ground_truth}","query_id":"${data.query_id}"}},
    }
)
print("Full eval summary:", out)

[2025-04-25 14:56:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 14:56:22 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_ndcg_3_20250425_145621_978152, log path: /Users/marcjimz/.promptflow/.runs/azure_ai_evaluation_evaluators_ndcg_3_20250425_145621_978152/logs.txt
[2025-04-25 14:56:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 14:56:22 -0700][promptflow._core.entry_meta_generator][WARNING] - Generate meta in current process and timeout won't take effect. Please handle timeout manually outside current process.
[2025-04-25 14:56:22 -0700][promptflow._sdk._orchestrator.run_submitter][INFO] - Submitting run azure_ai_evaluation_evaluators_recall_

2025-04-25 14:56:22 -0700   13228 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-04-25 14:56:22 -0700   13228 execution.bulk     INFO     Current thread is not main thread, skip signal handler registration in BatchEngine.
2025-04-25 14:56:22 -0700   13228 execution.bulk     INFO     The timeout for the batch run is 3600 seconds.
2025-04-25 14:56:22 -0700   13228 execution.bulk     INFO     Current system's available memory is 8850.4375MB, memory consumption of current process is 282.984375MB, estimated available worker count is 8850.4375/282.984375 = 31
2025-04-25 14:56:22 -0700   13228 execution.bulk     INFO     Set process count to 4 by taking the minimum value among the factors of {'default_worker_count': 4, 'row_count': 140, 'estimated_worker_count_based_on_memory_usage': 31}.
2025-04-25 14:56:23 -0700   13228 execution.bulk     INFO     Process name(SpawnProcess-25)-Process id(13914)-Line number(0) start execut

In [10]:
out

{'rows': [{'inputs.query_id': 'q1',
   'inputs.query': "Crohn's disease Adalimumab prior authorization criteria",
   'inputs.method': 'hybrid-semantic',
   'inputs.ranking': '{"d40": 2.8734118938446045, "d52": 2.8266220092773438, "d32": 2.811025381088257, "d66": 2.753601312637329, "d17": 2.7202813625335693, "d14": 2.7110650539398193, "d51": 2.7039756774902344, "d18": 2.6671109199523926, "d2": 2.6607306003570557, "d71": 2.613231658935547, "d50": 2.594090223312378, "d34": 2.579911470413208, "d1": 2.557225465774536, "d41": 2.419337034225464, "d25": 2.4127793312072754, "d3": 2.397005558013916, "d7": 2.3759145736694336, "d15": 2.363330841064453, "d31": 2.335416316986084, "d44": 2.3316946029663086, "d21": 2.3296563625335693, "d67": 2.2875630855560303, "d8": 2.1701009273529053, "d60": 2.0933139324188232, "d23": 2.0787806510925293, "d33": 1.913864016532898, "d56": 1.89064621925354, "d48": 1.8397799730300903, "d61": 1.8128403425216675, "d57": 1.6710525751113892, "d27": 1.6526201963424683, "d28"